[![Roboflow Notebooks](https://media.roboflow.com/notebooks/template/bannertest2-2.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672932710194)](https://github.com/roboflow/notebooks)

# How to Train a Re-ID Model for Tracking

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roboflow/re-ID/blob/main/docs/cookbooks/how-to-train-reid-model-for-tracking.ipynb)

A tracker swaps identities when two players look alike. Fine-tuning a Re-ID model on crops cut from your own videos teaches it the differences between them, so the tracker keeps each player's ID. In this notebook we fine-tune the general pedestrian model on SportsMOT with batches drawn from one sequence, score it before and after training, and then track a held-out clip with BoT-SORT to see what changes.

> **Runtime:** T4 GPU.

## Setup

### Check GPU availability

Training runs on every crop many times, so use a GPU runtime. If the cell below fails, go to `Runtime` → `Change runtime type` and pick a GPU.

In [ ]:
!nvidia-smi

### Install `reid`, `trackers` and `rfdetr`

We use `trackers` to download SportsMOT and to track, and `rfdetr` to detect the players we track.

In [ ]:
!pip install -q "reid @ git+https://github.com/roboflow/re-ID@feat/training-plots" trackers rfdetr

### Imports

In [ ]:
from pathlib import Path

import cv2
import numpy as np
import supervision as sv
from IPython import display
from PIL import Image
from rfdetr import RFDETRMedium
from trackers import BoTSORTTracker, ReIDConfig
from trackers.eval import evaluate_mot_sequences

from reid import ReassociationProtocol, ReIDModel, load_identity_folders, plot_training_history
from reid.data.mot import generate_mot_patches

## Download SportsMOT

SportsMOT is a multi-object tracking benchmark of basketball, football and volleyball broadcasts, where every player has a box and an identity in every frame. Each sequence is a clip of a longer video, and the part of its name before `_c` names the video, so `v_00HRwkvvjtQ_c001` and `v_00HRwkvvjtQ_c003` are two clips of one game. The train and val splits hold clips of different videos. The test split comes without annotations, so we download only train and val, about 13 GB of frames.

In [ ]:
!trackers download --name sportsmot --split train,val --asset annotations,frames

## Convert tracking data into a Re-ID dataset

`generate_mot_patches` crops every ground-truth box into one folder per track, and `load_identity_folders` reads those folders back as a `ReIDSplit` that labels each crop with its track's identity.

We cut three sets of crops. `train/` holds all of `sportsmot/train`, and the model trains on it. We split the videos of `sportsmot/val` in two, alternating between the halves in name order. `val/` holds the clips of the first half and is the validation set, which `train` scores after every epoch to pick the best one. `test/` holds the clips of the second half and is used only for scoring. We split by video rather than by clip because clips of one video show the same players under different identities, so a split by clip would test the model on players it already saw in validation.

In [ ]:
# Split the val videos in two, alternating in name order, and keep the clips of each half
all_clips = sorted(path.name for path in Path("sportsmot/val").iterdir() if (path / "gt").is_dir())
videos = sorted({clip.rsplit("_c", 1)[0] for clip in all_clips})
val_videos = videos[0::2]
val_clips = [clip for clip in all_clips if clip.rsplit("_c", 1)[0] in val_videos]
test_clips = [clip for clip in all_clips if clip.rsplit("_c", 1)[0] not in val_videos]

# Crop each set once, so running the cell again reuses the folders
cuts = [("sportsmot/train", "train", None), ("sportsmot/val", "val", val_clips), ("sportsmot/val", "test", test_clips)]
for mot_dir, crops_dir, clips in cuts:
    if not Path(crops_dir).is_dir():
        generate_mot_patches(mot_dir, crops_dir, split="full", sequences=clips)

train_crops = load_identity_folders("train")
val_crops = load_identity_folders("val")
test_crops = load_identity_folders("test")

for name, crops in (("train", train_crops), ("val", val_crops), ("test", test_crops)):
    print(f"{name}: {len(crops.image_paths)} crops of {len(np.unique(crops.pids))} players")

## Score the model before training

We start from `osnet_x1_0_msmt17_combineall`, the general pedestrian model, and score it on `test/` with `ReassociationProtocol`. For each box, it ranks the boxes 30 frames later by how close their embeddings are, as a tracker does when it links a player across frames. `rank1` is the percentage of boxes whose closest match is the right one, and `mrr` averages 1 / rank of the right one.

In [ ]:
model = ReIDModel.from_pretrained("osnet_x1_0_msmt17_combineall")
protocol = ReassociationProtocol()

before = model.evaluate(test_crops, protocol=protocol, verbose=False)
print(f"rank1 {before.metrics['rank1']:.1f}  mrr {before.metrics['mrr']:.1f}")

## Train

Every player is cropped in every frame, so `train/` holds over 300,000 crops, and neighboring frames give nearly the same image. With `steps_per_epoch=1000`, each epoch is 1000 batches drawn from all the crops instead of one pass over them, so the 20 epochs see each crop about twice.

The triplet loss learns to separate each player from the others in the batch. With batches from the whole set, those others are mostly players from other games, and the same player can appear under the IDs of two clips of one game. `sample_per_sequence=True` builds each batch from one sequence, so the triplet loss compares each player with the players in the same clip, which are the ones a tracker can mix up.

With `val_data`, `train` scores the model on `val/` before training and after every epoch, and saves the epoch with the highest MRR to `best/` in `output_dir`. It also plots the embeddings of 10 validation players after every epoch, under that epoch's line. `num_workers=2` loads and augments the crops in 2 background processes.

In [ ]:
history = model.train(
    train_crops,
    output_dir="runs/sportsmot",
    val_data=val_crops,
    val_protocol=ReassociationProtocol(),
    epochs=20,
    steps_per_epoch=1000,
    sample_per_sequence=True,
    num_workers=2,
)

## Plot the training curves

To plot the losses, the validation scores and the learning rate by epoch, we use `plot_training_history`.

In [ ]:
fig = plot_training_history("runs/sportsmot")

Validation MRR jumps from 68.9 to 84.1 at epoch 4, when the backbone starts training. After that it rises unevenly, moving up and down by 1 to 2 points between epochs. It peaks at 87.7 at epoch 19, marked by the dashed line, and drops to 86.1 at the last epoch even though the learning rate was already 3e-6. Keeping the best epoch in `best/` protects against drops like this one.

`output_dir` holds `last/`, the model after the last epoch, and `best/`, the epoch with the highest MRR on `val/`. Both load with `from_pretrained` like any model. Next to them, `metrics.csv` holds the rows of `history`, `training_config.json` the run's settings, and `embeddings/` one plot per epoch.

In [ ]:
print(sorted(path.name for path in Path("runs/sportsmot").iterdir()))

## Score the model after training

We score the last epoch, which `model` holds, and the best epoch, loaded from `best/`, on `test/`, which neither training nor validation saw. The protocol picks the same queries every time, so the scores compare the models on identical inputs.

In [ ]:
best_model = ReIDModel.from_pretrained("runs/sportsmot/best")

results = {
    "before training": before,
    "last epoch": model.evaluate(test_crops, protocol=protocol, verbose=False),
    "best epoch": best_model.evaluate(test_crops, protocol=protocol, verbose=False),
}
print(f"{'':<18}{'rank1':>8}{'mrr':>8}")
for name, result in results.items():
    print(f"{name:<18}{result.metrics['rank1']:>8.1f}{result.metrics['mrr']:>8.1f}")

## Track the players

Now we can try the trained model for tracking. We track one test clip, `v_5ekaksddqrc_c002`, with BoT-SORT on RF-DETR detections, without Re-ID, with the general model and with the trained one from `best/`.

In [ ]:
clip = "v_5ekaksddqrc_c002"
detector = RFDETRMedium()
general_model = ReIDModel.from_pretrained("osnet_x1_0_msmt17_combineall")
trained_model = ReIDModel.from_pretrained("runs/sportsmot/best")
# The values the trackers Re-ID guide tuned for SportsMOT
reid_config = ReIDConfig(appearance_threshold=0.3425, proximity_threshold=0.9953)
trackers_by_name = {
    "boxes_only": BoTSORTTracker(frame_rate=25),
    "general": BoTSORTTracker(frame_rate=25, reid_model=general_model, reid_config=reid_config),
    "trained": BoTSORTTracker(frame_rate=25, reid_model=trained_model, reid_config=reid_config),
}

# Detect once per frame and feed the same boxes to every tracker
rows = {name: [] for name in trackers_by_name}
frame_paths = sorted(Path(f"sportsmot/val/{clip}/img1").glob("*.jpg"))
for frame_path in frame_paths:
    frame = cv2.imread(str(frame_path))
    detections = detector.predict(frame[:, :, ::-1].copy(), threshold=0.1)  # RF-DETR takes RGB
    detections = detections[detections.class_id == 1]  # people, COCO class 1
    for name, tracker in trackers_by_name.items():
        tracks = tracker.update(detections, frame)
        for (x1, y1, x2, y2), track_id, confidence in zip(tracks.xyxy, tracks.tracker_id, tracks.confidence):
            if track_id >= 0:  # -1 marks a track not confirmed yet
                box = f"{x1:.1f},{y1:.1f},{x2 - x1:.1f},{y2 - y1:.1f}"
                rows[name].append(f"{int(frame_path.stem)},{track_id},{box},{confidence:.3f},-1,-1,-1")

# One MOT-format file per tracker
for name, lines in rows.items():
    Path(f"tracks/{name}").mkdir(parents=True, exist_ok=True)
    Path(f"tracks/{name}/{clip}.txt").write_text("\n".join(lines) + "\n")
print(f"{clip}: {len(frame_paths)} frames")

We score the tracks against SportsMOT's ground truth with `evaluate_mot_sequences` from `trackers`. HOTA weighs finding the players and keeping their identities together, AssA scores only how well the identities hold, and IDF1 is the share of boxes matched to the right player over the whole clip.

In [ ]:
Path("tracks/seqmap.txt").write_text(clip + "\n")
labels = {"boxes_only": "no Re-ID", "general": "general model", "trained": "trained model"}

print(f"{'':<20}{'HOTA':>8}{'IDF1':>8}{'AssA':>8}")
for name, label in labels.items():
    result = evaluate_mot_sequences(
        gt_dir="sportsmot/val",
        tracker_dir=f"tracks/{name}",
        seqmap="tracks/seqmap.txt",
        metrics=["HOTA", "Identity"],
    )
    hota, identity = result.aggregate.HOTA, result.aggregate.Identity
    print(f"{label:<20}{hota.HOTA * 100:>8.1f}{identity.IDF1 * 100:>8.1f}{hota.AssA * 100:>8.1f}")

With the general model, BoT-SORT barely improves on boxes alone, from 52.7 to 54.4 HOTA. The trained model raises HOTA to 66.1, IDF1 from 57.6 to 72.2 and AssA from 43.1 to 63.3, so most of the gain comes from keeping identities. This is one clip, so the numbers move more than they would over the whole test set.

In the GIF below, around frame 225, the general model loses the player with the ball as she drives past the defense, and when she reappears it gives her #0 and moves #11 to her teammate. The trained model keeps #11 on her the whole time. `SWAP_FRAMES` and `SHOWN_IDS` hold the frames and track IDs from our run, and a new run can number the tracks differently, so check them against your own tracks in `tracks/`.

In [ ]:
SWAP_FRAMES = range(196, 257, 2)  # every other frame around the swap
SHOWN_IDS = {"general": (0, 11), "trained": (0, 11)}  # each model's track IDs of the two players

color = sv.ColorPalette.from_hex(
    [
        "#ffff00",
        "#ff9b00",
        "#ff8080",
        "#ff66b2",
        "#ff66ff",
        "#b266ff",
        "#9999ff",
        "#3399ff",
        "#66ffff",
        "#33ff99",
        "#66ff66",
        "#99ff00",
    ]
)
box_annotator = sv.BoxAnnotator(color=color, color_lookup=sv.ColorLookup.TRACK, thickness=3)
label_annotator = sv.LabelAnnotator(
    color=color, color_lookup=sv.ColorLookup.TRACK, text_color=sv.Color.BLACK, text_scale=0.8
)

# Only the two players' tracks, from each model
shown = {}
for name, track_ids in SHOWN_IDS.items():
    rows = np.loadtxt(f"tracks/{name}/{clip}.txt", delimiter=",", ndmin=2)
    shown[name] = rows[np.isin(rows[:, 0], SWAP_FRAMES) & np.isin(rows[:, 1], track_ids)]

# One crop around both players in every frame, so no box or label is cut
height, width = cv2.imread(str(frame_paths[0])).shape[:2]
boxes = np.vstack(list(shown.values()))[:, 2:6]
left, top = np.maximum(boxes[:, :2].min(axis=0) - 60, 0).astype(int)
right, bottom = np.minimum((boxes[:, :2] + boxes[:, 2:4]).max(axis=0) + 60, (width, height)).astype(int)

gif_frames = []
for frame_number in SWAP_FRAMES:
    frame = cv2.imread(f"sportsmot/val/{clip}/img1/{frame_number:06d}.jpg")
    panels = []
    for name, rows in shown.items():
        in_frame = rows[rows[:, 0] == frame_number]
        detections = sv.Detections(
            xyxy=np.column_stack([in_frame[:, 2:4], in_frame[:, 2:4] + in_frame[:, 4:6]]),
            tracker_id=in_frame[:, 1].astype(int),
        )
        track_labels = [f"#{track_id}" for track_id in detections.tracker_id]
        panel = box_annotator.annotate(frame.copy(), detections)
        panel = label_annotator.annotate(panel, detections, labels=track_labels)[top:bottom, left:right]
        title = np.zeros((44, panel.shape[1], 3), dtype=np.uint8)
        cv2.putText(title, labels[name], (12, 32), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
        panels.append(np.vstack([title, panel]))
    gap = np.full((panels[0].shape[0], 8, 3), 255, dtype=np.uint8)
    gif_frames.append(Image.fromarray(cv2.cvtColor(np.hstack([panels[0], gap, panels[1]]), cv2.COLOR_BGR2RGB)))

# Every other frame of 25 frames per second, at half speed, and the middle third, where the IDs change hands, slower
third = len(gif_frames) // 3
durations = [160] * third + [320] * (len(gif_frames) - 2 * third) + [160] * third
gif_frames[0].save("tracking.gif", save_all=True, append_images=gif_frames[1:], duration=durations, loop=0)
display.Image(filename="tracking.gif")

You just fine-tuned a Re-ID model on crops cut from tracking video, with batches from one sequence, and tracked with it. Nice work! [Train a model for tracking](https://reid.roboflow.com/guides/training-for-tracking/) covers how long to make an epoch, why to draw each batch from one sequence and how to validate like a tracker.

Got feedback or ideas? Open an issue on [GitHub Issues](https://github.com/roboflow/re-ID/issues).